In [2]:
import pandas as pd
import numpy as np
import sqlite3

conn = sqlite3.connect("../database/yieldwatch.db")

runs = pd.read_sql("SELECT * FROM production_runs", conn)
sensors = pd.read_sql("SELECT * FROM sensor_readings", conn)
machines = pd.read_sql("SELECT * FROM machines", conn)

runs["timestamp"] = pd.to_datetime(runs["timestamp"])
sensors["timestamp"] = pd.to_datetime(sensors["timestamp"])
machines["installation_date"] = pd.to_datetime(machines["installation_date"])

# Merge sensors into runs — this becomes our main feature table
df = runs.merge(sensors.drop(columns=["timestamp"]), on="run_id")
df = df.merge(machines[["machine_id", "installation_date"]], on="machine_id")

print(df.shape)
df.columns.tolist()

(60000, 23)


['run_id',
 'timestamp',
 'line_id',
 'machine_id',
 'shift_id',
 'material_id',
 'operator_id',
 'planned_units',
 'produced_units',
 'good_units',
 'defective_units',
 'yield_rate',
 'defect_rate',
 'sensor_id',
 'temperature',
 'pressure',
 'vibration',
 'humidity',
 'motor_current',
 'rpm',
 'cycle_time',
 'tool_wear',
 'installation_date']

In [3]:
df["defect_flag"] = (df["defective_units"] > 0).astype(int)

print(df["defect_flag"].value_counts())
print("\n% of runs with at least one defect:", round(df["defect_flag"].mean() * 100, 2), "%")

defect_flag
1    59959
0       41
Name: count, dtype: int64

% of runs with at least one defect: 99.93 %


In [4]:
plant_avg_defect_rate = df["defect_rate"].mean()
print("Plant average defect rate:", round(plant_avg_defect_rate * 100, 2), "%")

df["defect_flag"] = (df["defect_rate"] > plant_avg_defect_rate).astype(int)

print(df["defect_flag"].value_counts())
print("\n% of runs flagged as high-defect:", round(df["defect_flag"].mean() * 100, 2), "%")

Plant average defect rate: 5.11 %
defect_flag
0    32974
1    27026
Name: count, dtype: int64

% of runs flagged as high-defect: 45.04 %


In [5]:
temp_mean = df["temperature"].mean()
vib_mean = df["vibration"].mean()

df["temperature_deviation"] = df["temperature"] - temp_mean
df["vibration_deviation"] = df["vibration"] - vib_mean

# Tool wear category — bucketing a continuous value into meaningful groups
df["tool_wear_category"] = pd.cut(
    df["tool_wear"],
    bins=[-1, 33, 66, 100],
    labels=["Low", "Medium", "High"]
)

print(df[["temperature", "temperature_deviation", "vibration", "vibration_deviation"]].head())
print("\nTool wear category distribution:")
print(df["tool_wear_category"].value_counts())

   temperature  temperature_deviation  vibration  vibration_deviation
0        79.50               5.025056       5.25             1.085585
1        73.17              -1.304944       2.76            -1.404415
2        70.81              -3.664944       5.33             1.165585
3        82.39               7.915056       6.54             2.375585
4        71.73              -2.744944       5.33             1.165585

Tool wear category distribution:
tool_wear_category
High      24929
Medium    19970
Low       15101
Name: count, dtype: int64


In [6]:
# Machine age at time of run (in years) - leakage-safe, known before the run happens
df["machine_age_years"] = (df["timestamp"] - df["installation_date"]).dt.days / 365

# Encode shift as a number (models need numeric input, not text)
shift_map = {"Morning": 0, "Evening": 1, "Night": 2}
df["shift_encoded"] = df["shift_id"].map(shift_map)

# One-hot encode material_id: turns each material into its own 0/1 column
material_dummies = pd.get_dummies(df["material_id"], prefix="material")
df = pd.concat([df, material_dummies], axis=1)

print(df[["machine_age_years", "shift_id", "shift_encoded"]].head())
print("\nNew material columns:", material_dummies.columns.tolist())

   machine_age_years shift_id  shift_encoded
0           3.383562  Morning              0
1           0.254795  Evening              1
2           4.027397    Night              2
3           3.805479    Night              2
4           3.347945  Evening              1

New material columns: ['material_MAT01', 'material_MAT02', 'material_MAT03', 'material_MAT04', 'material_MAT05', 'material_MAT06', 'material_MAT07', 'material_MAT08', 'material_MAT09', 'material_MAT10']


In [7]:
# Sort by machine and time - essential before calculating any "previous" or "rolling" feature
df = df.sort_values(["machine_id", "timestamp"]).reset_index(drop=True)

# Previous run's defect_rate for that SAME machine (shifted by 1 - excludes the current row)
df["prev_defect_rate"] = df.groupby("machine_id")["defect_rate"].shift(1)

# Rolling average of the last 5 runs for that machine, EXCLUDING the current run
df["rolling_5run_defect_rate"] = (
    df.groupby("machine_id")["defect_rate"]
    .shift(1)  # shift first, so "rolling" never includes the current row
    .rolling(5)
    .mean()
    .reset_index(level=0, drop=True)
)

# The very first run for each machine has no history - fill with the overall plant average
df["prev_defect_rate"] = df["prev_defect_rate"].fillna(plant_avg_defect_rate)
df["rolling_5run_defect_rate"] = df["rolling_5run_defect_rate"].fillna(plant_avg_defect_rate)

print(df[["machine_id", "timestamp", "defect_rate", "prev_defect_rate", "rolling_5run_defect_rate"]].head(10))

  machine_id  timestamp  defect_rate  prev_defect_rate  \
0        M01 2025-09-01     0.062147          0.051129   
1        M01 2025-09-01     0.036585          0.062147   
2        M01 2025-09-01     0.022358          0.036585   
3        M01 2025-09-01     0.020595          0.022358   
4        M01 2025-09-01     0.076233          0.020595   
5        M01 2025-09-01     0.062954          0.076233   
6        M01 2025-09-01     0.065455          0.062954   
7        M01 2025-09-01     0.087719          0.065455   
8        M01 2025-09-01     0.022959          0.087719   
9        M01 2025-09-02     0.024943          0.022959   

   rolling_5run_defect_rate  
0                  0.051129  
1                  0.051129  
2                  0.051129  
3                  0.051129  
4                  0.051129  
5                  0.043584  
6                  0.043745  
7                  0.049519  
8                  0.062591  
9                  0.063064  


In [8]:
df["temp_anomaly"] = (df["temperature"] > 80).astype(int)
df["vibration_anomaly"] = (df["vibration"] > 5).astype(int)
df["tool_wear_anomaly"] = (df["tool_wear"] > 75).astype(int)

print(df[["temp_anomaly", "vibration_anomaly", "tool_wear_anomaly"]].sum())
print("\n% of runs with any anomaly flag:",
      round(((df["temp_anomaly"] + df["vibration_anomaly"] + df["tool_wear_anomaly"]) > 0).mean() * 100, 2), "%")

temp_anomaly          9428
vibration_anomaly    14656
tool_wear_anomaly    19495
dtype: int64

% of runs with any anomaly flag: 54.88 %


In [9]:
feature_columns = [
    "temperature", "vibration", "tool_wear", "pressure", "humidity", "motor_current", "rpm", "cycle_time",
    "temperature_deviation", "vibration_deviation",
    "temp_anomaly", "vibration_anomaly", "tool_wear_anomaly",
    "machine_age_years", "shift_encoded",
    "prev_defect_rate", "rolling_5run_defect_rate"
] + material_dummies.columns.tolist()

print("Total features:", len(feature_columns))
print(feature_columns)

# Save the full feature-engineered dataset for Phase 10
model_ready = df[["run_id", "defect_flag"] + feature_columns].copy()
model_ready.to_csv("../data/processed/model_ready_features.csv", index=False)
print("\nSaved model_ready_features.csv:", model_ready.shape)

Total features: 27
['temperature', 'vibration', 'tool_wear', 'pressure', 'humidity', 'motor_current', 'rpm', 'cycle_time', 'temperature_deviation', 'vibration_deviation', 'temp_anomaly', 'vibration_anomaly', 'tool_wear_anomaly', 'machine_age_years', 'shift_encoded', 'prev_defect_rate', 'rolling_5run_defect_rate', 'material_MAT01', 'material_MAT02', 'material_MAT03', 'material_MAT04', 'material_MAT05', 'material_MAT06', 'material_MAT07', 'material_MAT08', 'material_MAT09', 'material_MAT10']

Saved model_ready_features.csv: (60000, 29)
